# Softmax Regression: From Math to Code

While linear regression is used to answer "how much?" (predicting continuous values), **classification** is used to answer "which category?" (predicting discrete classes). Softmax Regression is the foundational algorithm for multi-class classification in machine learning. 

Colloquially, classification models map input features to a set of probabilities, allowing us to make "soft assignments" (e.g., predicting an image is 80% likely to be a cat and 20% likely to be a dog) rather than just hard assignments.

In this notebook, we will build a Softmax Regression model from scratch using NumPy. We will use the textbook's example of classifying an image into one of three categories: **Cat, Chicken, and Dog**. To make the math easy to visualize, we will assume our inputs have been compressed into just $2$ features (e.g., "fluffiness" and "weight").

In [1]:
import numpy as np# Set print options for cleaner output
np.set_printoptions(precision=4, suppress=True)

## 1. Classification & One-Hot Encoding

Categorical data lacks a natural numerical order (a dog is not "greater" than a cat). To represent these categories mathematically, we use **one-hot encoding**. A one-hot encoding is a vector with as many components as we have categories, where the correct category is set to $1$ and all others to $0$.

Let's define our classes as {Cat, Chicken, Dog}. Our label $y$ would be a three-dimensional vector:

$$y \in \{(1, 0, 0), (0, 1, 0), (0, 0, 1)\}$$

Let's create the true labels for a small dataset of 3 images: a Cat, a Dog, and another Cat.

In [2]:
Y_true=np.array([
    [1,0,0],#Cat
    [0,1,0],#Dog
    [1,0,0]#Cat
])
print("True Labels (One-Hot Encoded):\n", Y_true)

True Labels (One-Hot Encoded):
 [[1 0 0]
 [0 1 0]
 [1 0 0]]


## 2. The Linear Model (Unvectorized & Vectorized)

To estimate the probabilities associated with all possible classes, we need a model with multiple outputs. We calculate a raw score (called a "logit") for each class using an affine function (weights and biases).

For a single example with 4 features and 3 outputs, the unvectorized equations look like this:
$$o_1 = x_1w_{11} + x_2w_{12} + x_3w_{13} + x_4w_{14} + b_1$$
$$o_2 = x_1w_{21} + x_2w_{22} + x_3w_{23} + x_4w_{24} + b_2$$
$$o_3 = x_1w_{31} + x_2w_{32} + x_3w_{33} + x_4w_{34} + b_3$$

To improve computational efficiency, we vectorize these calculations to process a minibatch of $n$ examples simultaneously using matrix multiplication:
$$\mathbf{O} = \mathbf{X}\mathbf{W} + \mathbf{b}$$

Let's define our inputs $\mathbf{X}$ (3 images, 2 features) and initialize random weights $\mathbf{W}$ and biases $\mathbf{b}$.

In [3]:
#minibatch of 3 images, each with 2 features
X=np.array([
    [2.0,5.0],
    [1.0,8.0],
    [2.2,4.8]
])

#weights :2x3 2 features and 3 classes
W=np.array([
    [0.1,0.5,-0.2],
    [-0.1,0.2,0.4]
])
#biases : 1x3
b=np.array([0.0,0.1,-0.1])

#calculate the linear output,size:3x3
O=np.dot(X,W)+b
print("Linear Output (Logits):\n", O)

Linear Output (Logits):
 [[-0.3   2.1   1.5 ]
 [-0.7   2.2   2.9 ]
 [-0.26  2.16  1.38]]


## 3. The Softmax Function

The raw scores $\mathbf{O}$ can be negative and do not sum to $1$. To convert them into valid probability distributions, we use the **softmax function**. It exponentiates the scores to make them positive, then divides by the sum to normalize them:

$$\hat{y}_i = \frac{\exp(o_i)}{\sum_j \exp(o_j)}$$

Because the exponential function is monotonic, it preserves the ordering of the arguments. Therefore, to just find the most likely class, we don't strictly need to compute the softmax; we can just take the argmax of the raw scores:
$$\text{argmax}_j \hat{y}_j = \text{argmax}_j o_j$$

In [4]:
def softmax(logits):
    exp_logits=np.exp(logits)
    #normalize by diving by the sum of each row
    sum_exp=np.sum(exp_logits, axis=1, keepdims=True)
    return exp_logits/sum_exp

Y_pred=softmax(O)
print("Predicted Probabilities (Y_hat):\n", Y_pred)
print("\nVerify sums to 1 for each image:\n", np.sum(Y_pred, axis=1))


Predicted Probabilities (Y_hat):
 [[0.0553 0.6099 0.3347]
 [0.0179 0.3259 0.6562]
 [0.0575 0.6463 0.2963]]

Verify sums to 1 for each image:
 [1. 1. 1.]


## 4. Cross-Entropy Loss

Now that we have predictions $\hat{\mathbf{y}}$, we need a loss function to measure how far they are from the true labels $\mathbf{y}$. We use maximum likelihood estimation, which minimizes the negative log-likelihood. 

Because we use one-hot encoding, the loss simplifies to the **cross-entropy loss**. For a single prediction, it measures the negative log of the probability assigned to the *true* class:

$$l(\mathbf{y}, \hat{\mathbf{y}}) = - \sum_{j=1}^q y_j \log \hat{y}_j$$

In information theory, this measures the expected "surprisal" or the number of bits needed to encode the data relative to our model's predictions. We calculate this loss for each example in the minibatch and average it.

In [5]:
def cross_entropy_loss(Y_true, Y_pred):
    #add a tiny value to prevent log(0)
    epsilon=1e-15
    Y_pred_safe=np.clip(Y_pred,epsilon,1.0-epsilon)

    #calculate the loss for each sample
    individual_losses=-np.sum(Y_true*np.log(Y_pred_safe), axis=1)
    #return the average loss over the batch
    return np.mean(individual_losses),individual_losses
avg_loss,batch_losses=cross_entropy_loss(Y_true,Y_pred)
print("Loss for each image in batch:\n", batch_losses)
print(f"\nAverage Minibatch Cross-Entropy Loss: {avg_loss:.4f}")

Loss for each image in batch:
 [2.8944 1.1213 2.8565]

Average Minibatch Cross-Entropy Loss: 2.2907


## 5. The Monolithic Loss Equation & The Chain Rule

To truly understand how a neural network updates, we need to look at the **Total Cost Function** ($J$). This is what happens when we combine our entire forward pass—the linear model, the softmax function, and the cross-entropy loss—into one massive equation, averaged over a minibatch of $n$ examples.

For a specific example $\mathbf{x}^{(d)}$ and its true one-hot label $\mathbf{y}^{(d)}$, the total loss across the batch is:

$$J(\mathbf{W}, \mathbf{b}) = \frac{1}{n} \sum_{d=1}^n \left[ - \sum_{i=1}^q y_i^{(d)} \log \left( \frac{\exp(\mathbf{x}^{(d)} \cdot \mathbf{w}_i + b_i)}{\sum_{k=1}^q \exp(\mathbf{x}^{(d)} \cdot \mathbf{w}_k + b_k)} \right) \right]$$

### Breaking it down to update a weight ($w_{ij}$)



We want to update a single specific weight, let's call it $w_{ij}$ (the weight connecting feature $j$ to class $i$). To do this, we need to find the derivative of that massive Cost Function with respect to our specific weight: $\frac{\partial J}{\partial w_{ij}}$.

Because the equation is nested, we use the **Chain Rule** from calculus to crack it open from the outside in:

$$\frac{\partial J}{\partial w_{ij}} = \frac{\partial J}{\partial o_i} \times \frac{\partial o_i}{\partial w_{ij}}$$

**Part 1: The Outer Error ($\frac{\partial J}{\partial o_i}$)**
This is the derivative of the Loss with respect to the raw score (logit) $o_i$. As the textbook showed in Equation (4.1.10), the derivative of Softmax + Cross-Entropy miraculously simplifies to just **Prediction minus Reality**:
$$\frac{\partial J}{\partial o_i} = \hat{y}_i - y_i$$

**Part 2: The Inner Derivative ($\frac{\partial o_i}{\partial w_{ij}}$)**
This asks: how much did weight $w_{ij}$ affect the raw score $o_i$? Since the raw score equation is just $o_i = x_1w_{i1} + x_jw_{ij} + \dots + b_i$, the derivative with respect to $w_{ij}$ is simply the input feature it was attached to:
$$\frac{\partial o_i}{\partial w_{ij}} = x_j$$

**Putting it Together (The Gradient):**
Multiply them together, and the gradient for a specific weight is just the error of the class multiplied by the input feature!
$$\text{Gradient for } w_{ij} = (\hat{y}_i - y_i) \cdot x_j$$

### The Final Update Rule
To update the weight, we take the old weight and subtract a small fraction (the learning rate $\alpha$) of the average gradient across the batch:

$$w_{ij}^{(\text{new})} = w_{ij}^{(\text{old})} - \alpha \left[ \frac{1}{n} \sum_{d=1}^n (\hat{y}_i^{(d)} - y_i^{(d)}) x_j^{(d)} \right]$$

In code, we use matrix multiplication (`np.dot`) to do this math ## The Matrix Magic: `np.dot(X.T, dO)`
In code, we don't use slow `for` loops to calculate this summation. We use matrix multiplication (`np.dot`) to do this math for *all* weights and *all* examples simultaneously. 

By transposing the input matrix $\mathbf{X}$ (flipping it so rows become features and columns become images) and taking the dot product with the error matrix $\mathbf{dO}$, linear algebra automatically multiplies every input feature $x_j$ by every class error $(\hat{y}_i - y_i)$ across all images, and sums them up perfectly. It is the exact equivalent of the $\sum$ equation above, computed instantly for the entire network.

In [ ]:
#calculate the gradient of the loss with respect to the logits
dO=Y_pred-Y_true
print("Gradient of Logits (dO):\n", dO)

#calculate the gradient for the weights 
n_samples=X.shape[0]
dW=np.dot(X.T,dO)/n_samples
print("Gradient for Weights (dW):\n", dW)

#update the weights using a learning rate
learning_rate = 0.1
W_new = W - (learning_rate * dW)

print("\nOld Weights:\n", W)
print("\nNew Updated Weights:\n", W_new)
print("\nNotice how weights connected to correct predictions increase, while others decrease!")

Gradient of Logits (dO):
 [[-0.9447  0.6099  0.3347]
 [ 0.0179 -0.6741  0.6562]
 [-0.9425  0.6463  0.2963]]
Gradient for Weights (dW):
 [[-1.315   0.6558  0.6591]
 [-3.0347  0.2529  2.7818]]

Old Weights:
 [[ 0.1  0.5 -0.2]
 [-0.1  0.2  0.4]]

New Updated Weights:
 [[ 0.2315  0.4344 -0.2659]
 [ 0.2035  0.1747  0.1218]]

Notice how weights connected to correct predictions increase, while others decrease!
